# Registry tutorial
This tutorial will show you how to:
- understand `Registry` objects and their key role in DNAStream. 
- add entities to a registry. 
- use the Python dunder functions
- look up items in the registry by `label` or `id`. 
- use `io` functions to simplify insertation from `maf` and `csv` files.  
- update metadata of registered entities. 
- activate and deactivate registered entities. 
- query the registry with a label/id selector or selector function. 
- extract the registry or a subset of the registry to a `pandas.Dataframe`. 

In [19]:
# load packages
import os
from dnastream import DNAStream
myfile = "myfile.h5"

#if DNAstream file does not already exist, then create it
if not os.path.isfile(myfile):

    #A specialized DNAStream HDF5 file is created with built-in datasets
    ds = DNAStream.create(path=myfile, patient_id="patientX", verbose=True)
    #Create returns a DNAStream

    #use `close()` to close the stream when finished.
    ds.close()

## `Registry` objects and their role in DNAStream
Registries play a key role in DNAStream because they provide tracking and management of key research entities, such as samples, cells, SNVs, indels, SNPs. When multiple analysts are working on a project, it can be difficult to track which samples pass QA and are to be used indownstream analysis or what SNVs are the current active set when multiple SNV callers are run. It can be hard to manage metadata and easily format it to streamline bash and workflow scripts/pipelines. They also provide link to various measured data and computed analysis results. 

Formally, a registry is a structured 1-d append only dataset (think dataframe) stored in the DNAStream HDF5 file for perpetuity. Any modifications to the registry dataset are logged so registry event histories can be extracted. 

When streaming a DNAStream file, `Registry` objects serve as interfaces to the registry dataset. Via `Registry` objects, one can:   
- `add` new entities for registration
- query the registry to `get` a subset meeting selector critera, 
- `update` the metadata of editable fields
- `activate` entities by `id` or `label`
- `find` all entities by a given `label`
- extract all or part of the registry dataset `to_dataframe`


### The registry spine
Every registry dataset automatically includes the following eight fields:
1. id : this is a unique identifier (UUIDv4) string for every registered entity.
2. label : this an internally generated **non-unique** human-readable string used to identify entities. The schema will specify how this field is generated. 
3. idx : this is a internal index of the entity in the dataset
4. active : this is a boolean indicating if the entity is active. Registry policy is to allow at most one unique label to be active. 
5. created_at : timestamp (ISO8601 Z) the entity was registered
6. created_by : user that registered the entity
7. modified_at : timestamp (ISO8601 Z) the entity was last modified
8. modified_by : the user that last modified the entity.

*NOTE: Registry policy does not allow modification of registry spine fields.  It also prevents updated any fields used to generate the label field.*


### Accessing `Registry` objects
`DNAStream` currently includes three built-in registries. 
- `sample`
- `variant`
- `snp`

See the built-in schema documentation for more details on the fields these included. 

They can be accessed as attributes from a `DNAStream` object or by name via the `registry(name)` function within DNAStream.

In [20]:
with DNAStream.open(myfile, mode="r") as ds:
    #attribute access
    sample_reg = ds.sample
    variant_reg = ds.variant
    print(ds.snp)

    #registry acess function
    sample_reg = ds.registry("sample")
    print(sample_reg)
    print(ds.sample)


/registry/snp (Shape: (0,))
/registry/sample (Shape: (4,))
/registry/sample (Shape: (4,))


## Add entities to the registry
There are multiple options to register entities. Registry items can be added directly to a `Registry` object via the low-level `Registry.add(...)` function from a list of dictionaries, where each dictionary is an entity.  

But there is also an `io` class to help streamline insertation from standard formats ('maf' files) and CSV and TSV files. 

### Low-level insertion
To utilize low-level insertation, first obtain a referenece to a 'Registry' object and then call the `add(...)` function on a list of dictionaries.

In [21]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    reg.add([
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ])

    print(f"After add, sample registry contains {len(reg)} entities.")

Sample registry contains 4 entities.
After add, sample registry contains 4 entities.


There are two key parameters to the `add(...)` function which deal with the fact that internal `labels` are not required to be unique:
- `allow_duplicate_labels` : whether to allow the insertation to proceed with inserting new records that collide with existng registere entities (default is False).
- `activate_newest` : in the event of duplicate labels, this specifies whether to activate the new entity upon insertation (and deactivate the old entity) or whether to leave the newly inserted colliding entity deactivated.

In [22]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    reg.add([
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ], allow_duplicate_labels=True, activate_newest=True)

    print(f"After add, sample registry contains {len(reg)} entities.")

Sample registry contains 4 entities.
After add, sample registry contains 6 entities.


Now we can iterate through all of the sample entities in the dataset and view the following fields `label`, `id`, `active`, `sample_name`, `created_at`. Note that there are two copies of each label but only newest inserted entity for that label is activated.

In [23]:
with DNAStream.open(path=myfile) as ds:
    for sample in ds.sample:
        print(f'label: {sample["label"]}, id: {sample["id"]}, active: {sample["active"]}, sample_name {sample["sample_name"]}, created at:{sample["created_at"]}')

label: S1, id: 2bb82b81-3fe2-4a57-b40f-4140a8351c15, active: False, sample_name S1, created at:2026-02-04T20:31:12.618937Z
label: S2, id: a45f0243-04ce-47d6-94c9-565c468a1ce9, active: False, sample_name S2, created at:2026-02-04T20:31:12.618937Z
label: S1, id: d2b13d93-ca83-49d7-bb2b-bf8fa6488730, active: False, sample_name S1, created at:2026-02-04T20:37:31.415659Z
label: S2, id: f2529886-3c7e-49da-8381-726ba6043f83, active: False, sample_name S2, created at:2026-02-04T20:37:31.415659Z
label: S1, id: 756eefda-6ecb-423e-81f8-291a34cc7852, active: True, sample_name S1, created at:2026-02-04T20:57:08.035720Z
label: S2, id: fa3d8ba4-97e8-4986-9ae5-d283774b85ce, active: True, sample_name S2, created at:2026-02-04T20:57:08.035720Z


## DNAstream I/O functions
Typically, users will want to register a large number of entities from the output files of other upstream methods, like variant callers (maf files) or via CSV and TSV files.  `DNAStream` provides `io` wrappers to streamline the insertation of entities to built-in registries from files. 

Currently, `DNAStream` offers three `io` functions to add facilitate this:
1. `io.add_variants_from_maf`: add variants to the variant registry from a maf file or a list of maf files.
2. `io.add_snps_from_maf` : add SNPSs to the SNP registry from a maf file or a list of maf files.
3. `io.add_samples_from_files` : add samples to the sample registry from a CSV (default) or other delimited file.

Additionally, `DNAStream` offers static functions to parse CSV (`io.parse_csv`, `io.parse_tsv`) or TSV files for preparation in the list of dictionary format required by `Registry.add(...)`. 

In [24]:
#TODO write examples 

## Look up registered entities by `label` or `id`
The entity id is the only unique identifier for a registered entity. As described, labels are used to provide a standardized but non-unique identifier for an entity. Therefore, it is neccessary to convert from a label to and id or from id to a label. There are multiple functions to support this conversion:
- `resolve_ids` : given ids, resolve the 'activated' labels
- `resolved_labels` : given labels, resolve the UUIDv4 id
- `find_ids`: given labels, find all ids assosciated with each label

## Update the metadata of registered entities
As described above, only certain fields in a registry are modifiable. Registry datasets are only modifiable via the `Registry.update(...)` function. 